# Without Roots: The Political Consequences of Collective Economic Shocks

**Authors:** Cremaschi, Simone, Bariletto, Nicola, De Vries, Catherine E.

**Journal:** American Political Science Review (2025)

This notebook reproduces the analysis from the paper above.
It was auto-generated by [PoliRep](https://polirep.org).

---

## Setup

Install packages and import standard libraries.

In [ ]:
!pip install -q pandas numpy matplotlib scipy statsmodels pyfixest

import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

## Configuration & Constants

Paper-specific constants used by the analysis code below.

In [ ]:
# Paper-specific constants (extracted from config.py)

# Colab-friendly data path (replaces local PAPER_ROOT-based path)
DATA_CONVERTED = Path("data/converted")
OUTPUT_DIR = Path("output")
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"

XYLELLA_GROUP_EXCLUDE = 7
ELECTION_MONTHS = {
    2001: 5,
    2006: 4,
    2008: 4,
    2013: 2,
    2018: 3,
    2022: 9,
}
REFERENCE_YEAR = 2013
OUTCOMES_POLITICAL = [
    "farright_p",
    "right_p",
    "left_p",
    "farleft_p",
    "m5s_p",
    "turnout_p",
]
OUTCOMES_ECONOMIC = [
    "pretax_inc_pc",
    "arcsinh_pretax_pc",
]
OUTCOMES_SUICIDE = [
    "n_suicides",
    "suicide_arcsinh",
]
TREATMENT_VAR = "infected_dummy"
TREATMENT_INTENSITY = "intensity"
TREATED = "treated"
MUNICIPALITY_FE = "com_id"
YEAR_FE = "election_year"
PROVINCE = "prov"
CLUSTER_VAR = "xylella_group"

## Data Preparation


### Load National Elections Data

Load electoral data with party family aggregation and treatment variables.
Merges electoral_data.tab with party family crosswalk, municipal crosswalk,
and Xylella infection timing data.

In [ ]:
def load_electoral_data() -> pd.DataFrame:
    """Load electoral data with treatment variables.

    Note: This assumes data_processing.py from original/ has been run
    to create the cleaned datasets. If not, run that first.

    Returns:
        DataFrame: National elections dataset with treatment variables.
    """
    # Try to load cleaned data first
    cleaned_path = DATA_CONVERTED.parent / "cleaned" / "national_elections.parquet"

    if cleaned_path.exists():
        df = pd.read_parquet(cleaned_path)
        return df

    # Fall back to loading from converted raw data and processing
    # Load electoral data
    electoral = pd.read_parquet(DATA_CONVERTED / "electoral_data.parquet")

    # Load xylella infection data
    xylella = pd.read_parquet(DATA_CONVERTED / "xylella_areas.parquet")

    # Load crosswalks
    municipal_xwalk = pd.read_parquet(DATA_CONVERTED / "municipal_crosswalk.parquet")
    party_xwalk = pd.read_parquet(DATA_CONVERTED / "partyfam_crosswalk.parquet")

    # Merge and aggregate (simplified version - see data_processing.py for full logic)
    # This is a minimal implementation to get started
    df = electoral.merge(party_xwalk, on=["party", "election_year"], how="left")
    df = df.merge(municipal_xwalk, on="municipality", how="left")
    df = df.merge(xylella, on="com_id", how="left")

    # Create treatment variables
    df["treated"] = (df["xylella_group"] > 0).astype(int)
    df["infected_dummy"] = 0

    # Add election months
    df["election_month"] = df["election_year"].map(ELECTION_MONTHS)

    return df

In [ ]:
df = load_electoral_data()
print(f"Shape: {df.shape}")
print(f"Municipalities: {df['com_id'].nunique()}")
print(f"Elections: {df['election_year'].nunique()}")
print(f"Year range: {int(df['election_year'].min())}-{int(df['election_year'].max())}")
print(f"Party families: {sorted(df['partyfamily'].dropna().unique().tolist())}")
display(df[['com_id', 'election_year', 'partyfamily', 'treated', 'xylella_group']].head(10))


### Create Treatment Variables

Create treatment indicators: treated (ever infected), infected_dummy
(infected by election date), and intensity (months since infection).
Add election months and calculate time-to-treatment for event studies.

In [ ]:
def load_electoral_data() -> pd.DataFrame:
    """Load electoral data with treatment variables.

    Note: This assumes data_processing.py from original/ has been run
    to create the cleaned datasets. If not, run that first.

    Returns:
        DataFrame: National elections dataset with treatment variables.
    """
    # Try to load cleaned data first
    cleaned_path = DATA_CONVERTED.parent / "cleaned" / "national_elections.parquet"

    if cleaned_path.exists():
        df = pd.read_parquet(cleaned_path)
        return df

    # Fall back to loading from converted raw data and processing
    # Load electoral data
    electoral = pd.read_parquet(DATA_CONVERTED / "electoral_data.parquet")

    # Load xylella infection data
    xylella = pd.read_parquet(DATA_CONVERTED / "xylella_areas.parquet")

    # Load crosswalks
    municipal_xwalk = pd.read_parquet(DATA_CONVERTED / "municipal_crosswalk.parquet")
    party_xwalk = pd.read_parquet(DATA_CONVERTED / "partyfam_crosswalk.parquet")

    # Merge and aggregate (simplified version - see data_processing.py for full logic)
    # This is a minimal implementation to get started
    df = electoral.merge(party_xwalk, on=["party", "election_year"], how="left")
    df = df.merge(municipal_xwalk, on="municipality", how="left")
    df = df.merge(xylella, on="com_id", how="left")

    # Create treatment variables
    df["treated"] = (df["xylella_group"] > 0).astype(int)
    df["infected_dummy"] = 0

    # Add election months
    df["election_month"] = df["election_year"].map(ELECTION_MONTHS)

    return df

In [ ]:
df = load_electoral_data()
print(f"Rows with treated=1: {int(df['treated'].sum())} ({df['treated'].mean():.1%})")
print(f"Unique treated municipalities: {df[df['treated']==1]['com_id'].nunique()}")
print(f"Xylella groups: {sorted(df['xylella_group'].unique().tolist())}")
print(f"Election months available: {'election_month' in df.columns}")


### Calculate Vote Shares

Aggregate votes by party family (farright, right, left, farleft, m5s)
and calculate vote shares as % of total eligible voters. Also compute
turnout as % of eligible voters.

In [ ]:
def load_electoral_data() -> pd.DataFrame:
    """Load electoral data with treatment variables.

    Note: This assumes data_processing.py from original/ has been run
    to create the cleaned datasets. If not, run that first.

    Returns:
        DataFrame: National elections dataset with treatment variables.
    """
    # Try to load cleaned data first
    cleaned_path = DATA_CONVERTED.parent / "cleaned" / "national_elections.parquet"

    if cleaned_path.exists():
        df = pd.read_parquet(cleaned_path)
        return df

    # Fall back to loading from converted raw data and processing
    # Load electoral data
    electoral = pd.read_parquet(DATA_CONVERTED / "electoral_data.parquet")

    # Load xylella infection data
    xylella = pd.read_parquet(DATA_CONVERTED / "xylella_areas.parquet")

    # Load crosswalks
    municipal_xwalk = pd.read_parquet(DATA_CONVERTED / "municipal_crosswalk.parquet")
    party_xwalk = pd.read_parquet(DATA_CONVERTED / "partyfam_crosswalk.parquet")

    # Merge and aggregate (simplified version - see data_processing.py for full logic)
    # This is a minimal implementation to get started
    df = electoral.merge(party_xwalk, on=["party", "election_year"], how="left")
    df = df.merge(municipal_xwalk, on="municipality", how="left")
    df = df.merge(xylella, on="com_id", how="left")

    # Create treatment variables
    df["treated"] = (df["xylella_group"] > 0).astype(int)
    df["infected_dummy"] = 0

    # Add election months
    df["election_month"] = df["election_year"].map(ELECTION_MONTHS)

    return df

In [ ]:
df = load_electoral_data()
print("Raw vote totals by party family:")
if 'partyfamily' in df.columns and 'votes' in df.columns:
  party_totals = df.groupby('partyfamily')['votes'].sum().sort_values(ascending=False)
  for pf, votes in party_totals.items():
    print(f"  {pf}: {votes:,.0f}")
print(f"\nTotal rows: {len(df)}")


### Load Demographics and Suicide Data

Load demographics panel with suicide counts aggregated from death records
(ICD-10 codes X60-X84 for intentional self-harm). Merge with Xylella
infection timing.

In [ ]:
def load_demographics_data() -> pd.DataFrame:
    """Load demographics and suicide data.

    Returns:
        DataFrame: Demographics dataset with suicide counts.
    """
    cleaned_path = DATA_CONVERTED.parent / "cleaned" / "demographics.parquet"

    if cleaned_path.exists():
        return pd.read_parquet(cleaned_path)

    # Fall back to raw data
    demographics = pd.read_parquet(DATA_CONVERTED / "demographics_istat.parquet")
    return demographics

In [ ]:
demo = load_demographics_data()
print(f"Shape: {demo.shape}")
if 'wave' in demo.columns:
  print(f"Year range: {int(demo['wave'].min())}-{int(demo['wave'].max())}")
if 'n_suicides' in demo.columns:
  print(f"Suicides: {demo['n_suicides'].sum()} total")
  print(f"Mean suicides per muni-year: {demo['n_suicides'].mean():.2f}")
display_cols = [c for c in ['com_id', 'wave', 'n_suicides', 'agegroup_20_35_ita'] if c in demo.columns]
if display_cols:
  display(demo[display_cols].head())


### Load Income Data

Load income panel with pre-tax income per capita from MEF tax records.
Calculate arcsinh transformation for robust analysis.

In [ ]:
def load_income_data() -> pd.DataFrame:
    """Load income data.

    Returns:
        DataFrame: Income dataset.
    """
    cleaned_path = DATA_CONVERTED.parent / "cleaned" / "income.parquet"

    if cleaned_path.exists():
        return pd.read_parquet(cleaned_path)

    # Fall back to raw data
    income = pd.read_parquet(DATA_CONVERTED / "income_istat.parquet")
    return income

In [ ]:
inc = load_income_data()
print(f"Shape: {inc.shape}")
if 'wave' in inc.columns:
  print(f"Year range: {int(inc['wave'].min())}-{int(inc['wave'].max())}")
if 'pretax_inc_pc' in inc.columns:
  print(f"Mean pretax_inc_pc: €{inc['pretax_inc_pc'].mean():.0f}")
display_cols = [c for c in ['com_id', 'wave', 'pretax_inc_pc', 'arcsinh_pretax_pc'] if c in inc.columns]
if display_cols:
  display(inc[display_cols].head())


### Load Time-Invariant Covariates

Load covariates from 2001/2011 censuses: demographics, economic structure,
agricultural land use, distance to public service hubs, and social capital
indicators. Calculate derived variables (shares, densities).

In [ ]:
def load_covariates_data() -> pd.DataFrame:
    """Load time-invariant covariates.

    Returns:
        DataFrame: Covariates dataset.
    """
    cleaned_path = DATA_CONVERTED.parent / "cleaned" / "covariates.parquet"

    if cleaned_path.exists():
        return pd.read_parquet(cleaned_path)

    # Fall back to raw data
    covariates = pd.read_parquet(DATA_CONVERTED / "covariates_census.parquet")
    return covariates

In [ ]:
cov = load_covariates_data()
print(f"Shape: {cov.shape}")
if 'dist_polo_2011' in cov.columns:
  print(f"Mean dist_polo_2011: {cov['dist_polo_2011'].mean():.1f} minutes")
if 'pop_11' in cov.columns:
  print(f"Mean pop_11: {cov['pop_11'].mean():.0f}")
if 'surf_oliv' in cov.columns:
  print(f"Mean surf_oliv: {cov['surf_oliv'].mean():.1f} hectares")
display_cols = [c for c in ['com_id', 'pop_11', 'dist_polo_2011', 'surf_oliv'] if c in cov.columns]
if display_cols:
  display(cov[display_cols].head())


### Apply Sample Restrictions

Filter main analysis sample: exclude xylella_group 7 (May 2019 infection,
after 2018 election), restrict to Puglia, and exclude municipalities with
corrupted voter data. Create time-to-treatment variable for event studies.

In [ ]:
def prepare_analysis_sample(df: pd.DataFrame) -> pd.DataFrame:
    """Apply sample restrictions and construct derived variables.

    Args:
        df: Raw loaded DataFrame from load_main_data().

    Returns:
        Analysis-ready DataFrame with all filters and transformations applied.
    """
    sample = df.copy()

    # Main exclusion: xylella_group 7 (May 2019 infection, after 2018 election)
    sample = sample[sample["xylella_group"] != XYLELLA_GROUP_EXCLUDE].copy()

    # Create time-to-treatment variable for event studies
    # For treated municipalities: years relative to infection
    # For never-treated: set to -1000 (never treated indicator)
    if "x_year" in sample.columns and "election_year" in sample.columns:
        sample["time_to_treatment"] = sample["election_year"] - sample["x_year"]
        sample.loc[sample["treated"] == 0, "time_to_treatment"] = -1000

    # Create province-by-year fixed effects for trends
    if "prov" in sample.columns and "election_year" in sample.columns:
        sample["prov_year"] = (
            sample["prov"].astype(str) + "_" + sample["election_year"].astype(str)
        )

    return sample

In [ ]:
df = load_main_data()
sample = prepare_analysis_sample(df)
print(f"Full data shape: {df.shape}")
print(f"Analysis sample shape: {sample.shape}")
print(f"Municipalities: {sample['com_id'].nunique() if 'com_id' in sample.columns else 'N/A'}")
print(f"Treated municipalities (unique): {sample[sample['treated']==1]['com_id'].nunique() if 'com_id' in sample.columns and 'treated' in sample.columns else 'N/A'}")
print(f"Elections: {sample['election_year'].nunique() if 'election_year' in sample.columns else 'N/A'}")
stats = get_sample_stats(sample)
print(f"\nSample stats:")
for key, val in stats.items():
  print(f"  {key}: {val}")


## Main DiD Analysis


### Main DiD Analysis

Two-way fixed effects DiD on five political outcomes (Table 1)

In [ ]:
def run():
    """Run all main text analyses."""
    ensure_output_dirs()

    print("Loading data...")
    electoral = load_electoral_data()
    sample = prepare_analysis_sample(electoral)

    print(f"Sample size: {len(sample)} observations")
    print(f"Municipalities: {sample['com_id'].nunique()}")

    # Table 1: Main DiD
    print("\n" + "=" * 60)
    print("Table 1: Main DiD Results")
    print("=" * 60)
    table1 = run_table1_did(sample)
    print(table1)

    # Save table
    save_table_tex(
        table1,
        TABLE_DIR / "table_1.tex",
        title="Main DiD Results: Political Outcomes",
        note="TWFE DiD with municipality and year fixed effects, province-specific time trends. Standard errors clustered by Xylella infection group.",
    )

    # Figure 2a: Event study - far-right
    print("\n" + "=" * 60)
    print("Figure 2a: Event Study - Far-Right Vote Share")
    print("=" * 60)
    run_figure2a_event_study_farright(sample)
    print(f"Saved: {FIGURE_DIR / 'figure_2a.pdf'}")

    # Figure 2b: Event study - far-right/right ratio
    print("\n" + "=" * 60)
    print("Figure 2b: Event Study - Far-Right Share Within Right Bloc")
    print("=" * 60)
    run_figure2b_event_study_farright_over_right(sample)
    print(f"Saved: {FIGURE_DIR / 'figure_2b.pdf'}")

    # Figure 3a: Income null (requires income data)
    print("\n" + "=" * 60)
    print("Figure 3a: Income Null Result")
    print("=" * 60)
    try:
        income = load_income_data()
        income_sample = prepare_analysis_sample(income)
        run_figure3a_income_null(income_sample)
        print(f"Saved: {FIGURE_DIR / 'figure_3a.pdf'}")
    except Exception as e:
        print(f"Warning: Could not generate Figure 3a: {e}")

    # Figure 4: HTE by public service distance
    print("\n" + "=" * 60)
    print("Figure 4: HTE by Public Service Distance")
    print("=" * 60)
    try:
        covariates = load_covariates_data()
        run_figure4_hte_public_service(sample, covariates)
        print(f"Saved: {FIGURE_DIR / 'figure_4.pdf'}")
    except Exception as e:
        print(f"Warning: Could not generate Figure 4: {e}")

    print("\n" + "=" * 60)
    print("Main text analysis complete!")
    print("=" * 60)

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## Event Study - Far-Right


### Event Study - Far-Right

Event study for far-right vote share showing parallel pre-trends (Figures 2a-2b)

In [ ]:
def run():
    """Run all main text analyses."""
    ensure_output_dirs()

    print("Loading data...")
    electoral = load_electoral_data()
    sample = prepare_analysis_sample(electoral)

    print(f"Sample size: {len(sample)} observations")
    print(f"Municipalities: {sample['com_id'].nunique()}")

    # Table 1: Main DiD
    print("\n" + "=" * 60)
    print("Table 1: Main DiD Results")
    print("=" * 60)
    table1 = run_table1_did(sample)
    print(table1)

    # Save table
    save_table_tex(
        table1,
        TABLE_DIR / "table_1.tex",
        title="Main DiD Results: Political Outcomes",
        note="TWFE DiD with municipality and year fixed effects, province-specific time trends. Standard errors clustered by Xylella infection group.",
    )

    # Figure 2a: Event study - far-right
    print("\n" + "=" * 60)
    print("Figure 2a: Event Study - Far-Right Vote Share")
    print("=" * 60)
    run_figure2a_event_study_farright(sample)
    print(f"Saved: {FIGURE_DIR / 'figure_2a.pdf'}")

    # Figure 2b: Event study - far-right/right ratio
    print("\n" + "=" * 60)
    print("Figure 2b: Event Study - Far-Right Share Within Right Bloc")
    print("=" * 60)
    run_figure2b_event_study_farright_over_right(sample)
    print(f"Saved: {FIGURE_DIR / 'figure_2b.pdf'}")

    # Figure 3a: Income null (requires income data)
    print("\n" + "=" * 60)
    print("Figure 3a: Income Null Result")
    print("=" * 60)
    try:
        income = load_income_data()
        income_sample = prepare_analysis_sample(income)
        run_figure3a_income_null(income_sample)
        print(f"Saved: {FIGURE_DIR / 'figure_3a.pdf'}")
    except Exception as e:
        print(f"Warning: Could not generate Figure 3a: {e}")

    # Figure 4: HTE by public service distance
    print("\n" + "=" * 60)
    print("Figure 4: HTE by Public Service Distance")
    print("=" * 60)
    try:
        covariates = load_covariates_data()
        run_figure4_hte_public_service(sample, covariates)
        print(f"Saved: {FIGURE_DIR / 'figure_4.pdf'}")
    except Exception as e:
        print(f"Warning: Could not generate Figure 4: {e}")

    print("\n" + "=" * 60)
    print("Main text analysis complete!")
    print("=" * 60)

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## Mechanism Tests


### Mechanism Tests

Testing economic mechanism: income and young residents (Figures 3a-3b)

In [ ]:
def run():
    """Run all main text analyses."""
    ensure_output_dirs()

    print("Loading data...")
    electoral = load_electoral_data()
    sample = prepare_analysis_sample(electoral)

    print(f"Sample size: {len(sample)} observations")
    print(f"Municipalities: {sample['com_id'].nunique()}")

    # Table 1: Main DiD
    print("\n" + "=" * 60)
    print("Table 1: Main DiD Results")
    print("=" * 60)
    table1 = run_table1_did(sample)
    print(table1)

    # Save table
    save_table_tex(
        table1,
        TABLE_DIR / "table_1.tex",
        title="Main DiD Results: Political Outcomes",
        note="TWFE DiD with municipality and year fixed effects, province-specific time trends. Standard errors clustered by Xylella infection group.",
    )

    # Figure 2a: Event study - far-right
    print("\n" + "=" * 60)
    print("Figure 2a: Event Study - Far-Right Vote Share")
    print("=" * 60)
    run_figure2a_event_study_farright(sample)
    print(f"Saved: {FIGURE_DIR / 'figure_2a.pdf'}")

    # Figure 2b: Event study - far-right/right ratio
    print("\n" + "=" * 60)
    print("Figure 2b: Event Study - Far-Right Share Within Right Bloc")
    print("=" * 60)
    run_figure2b_event_study_farright_over_right(sample)
    print(f"Saved: {FIGURE_DIR / 'figure_2b.pdf'}")

    # Figure 3a: Income null (requires income data)
    print("\n" + "=" * 60)
    print("Figure 3a: Income Null Result")
    print("=" * 60)
    try:
        income = load_income_data()
        income_sample = prepare_analysis_sample(income)
        run_figure3a_income_null(income_sample)
        print(f"Saved: {FIGURE_DIR / 'figure_3a.pdf'}")
    except Exception as e:
        print(f"Warning: Could not generate Figure 3a: {e}")

    # Figure 4: HTE by public service distance
    print("\n" + "=" * 60)
    print("Figure 4: HTE by Public Service Distance")
    print("=" * 60)
    try:
        covariates = load_covariates_data()
        run_figure4_hte_public_service(sample, covariates)
        print(f"Saved: {FIGURE_DIR / 'figure_4.pdf'}")
    except Exception as e:
        print(f"Warning: Could not generate Figure 4: {e}")

    print("\n" + "=" * 60)
    print("Main text analysis complete!")
    print("=" * 60)

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## Heterogeneous Effects by Public Services


### Heterogeneous Effects by Public Services

Treatment effect moderation by distance to public service hubs (Figure 4)

In [ ]:
def run():
    """Run all main text analyses."""
    ensure_output_dirs()

    print("Loading data...")
    electoral = load_electoral_data()
    sample = prepare_analysis_sample(electoral)

    print(f"Sample size: {len(sample)} observations")
    print(f"Municipalities: {sample['com_id'].nunique()}")

    # Table 1: Main DiD
    print("\n" + "=" * 60)
    print("Table 1: Main DiD Results")
    print("=" * 60)
    table1 = run_table1_did(sample)
    print(table1)

    # Save table
    save_table_tex(
        table1,
        TABLE_DIR / "table_1.tex",
        title="Main DiD Results: Political Outcomes",
        note="TWFE DiD with municipality and year fixed effects, province-specific time trends. Standard errors clustered by Xylella infection group.",
    )

    # Figure 2a: Event study - far-right
    print("\n" + "=" * 60)
    print("Figure 2a: Event Study - Far-Right Vote Share")
    print("=" * 60)
    run_figure2a_event_study_farright(sample)
    print(f"Saved: {FIGURE_DIR / 'figure_2a.pdf'}")

    # Figure 2b: Event study - far-right/right ratio
    print("\n" + "=" * 60)
    print("Figure 2b: Event Study - Far-Right Share Within Right Bloc")
    print("=" * 60)
    run_figure2b_event_study_farright_over_right(sample)
    print(f"Saved: {FIGURE_DIR / 'figure_2b.pdf'}")

    # Figure 3a: Income null (requires income data)
    print("\n" + "=" * 60)
    print("Figure 3a: Income Null Result")
    print("=" * 60)
    try:
        income = load_income_data()
        income_sample = prepare_analysis_sample(income)
        run_figure3a_income_null(income_sample)
        print(f"Saved: {FIGURE_DIR / 'figure_3a.pdf'}")
    except Exception as e:
        print(f"Warning: Could not generate Figure 3a: {e}")

    # Figure 4: HTE by public service distance
    print("\n" + "=" * 60)
    print("Figure 4: HTE by Public Service Distance")
    print("=" * 60)
    try:
        covariates = load_covariates_data()
        run_figure4_hte_public_service(sample, covariates)
        print(f"Saved: {FIGURE_DIR / 'figure_4.pdf'}")
    except Exception as e:
        print(f"Warning: Could not generate Figure 4: {e}")

    print("\n" + "=" * 60)
    print("Main text analysis complete!")
    print("=" * 60)

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## Case Illustrations


### Case Illustrations

Municipality-level trajectories of far-right voting over time (Figure 5)

In [ ]:
def run():
    """Run all main text analyses."""
    ensure_output_dirs()

    print("Loading data...")
    electoral = load_electoral_data()
    sample = prepare_analysis_sample(electoral)

    print(f"Sample size: {len(sample)} observations")
    print(f"Municipalities: {sample['com_id'].nunique()}")

    # Table 1: Main DiD
    print("\n" + "=" * 60)
    print("Table 1: Main DiD Results")
    print("=" * 60)
    table1 = run_table1_did(sample)
    print(table1)

    # Save table
    save_table_tex(
        table1,
        TABLE_DIR / "table_1.tex",
        title="Main DiD Results: Political Outcomes",
        note="TWFE DiD with municipality and year fixed effects, province-specific time trends. Standard errors clustered by Xylella infection group.",
    )

    # Figure 2a: Event study - far-right
    print("\n" + "=" * 60)
    print("Figure 2a: Event Study - Far-Right Vote Share")
    print("=" * 60)
    run_figure2a_event_study_farright(sample)
    print(f"Saved: {FIGURE_DIR / 'figure_2a.pdf'}")

    # Figure 2b: Event study - far-right/right ratio
    print("\n" + "=" * 60)
    print("Figure 2b: Event Study - Far-Right Share Within Right Bloc")
    print("=" * 60)
    run_figure2b_event_study_farright_over_right(sample)
    print(f"Saved: {FIGURE_DIR / 'figure_2b.pdf'}")

    # Figure 3a: Income null (requires income data)
    print("\n" + "=" * 60)
    print("Figure 3a: Income Null Result")
    print("=" * 60)
    try:
        income = load_income_data()
        income_sample = prepare_analysis_sample(income)
        run_figure3a_income_null(income_sample)
        print(f"Saved: {FIGURE_DIR / 'figure_3a.pdf'}")
    except Exception as e:
        print(f"Warning: Could not generate Figure 3a: {e}")

    # Figure 4: HTE by public service distance
    print("\n" + "=" * 60)
    print("Figure 4: HTE by Public Service Distance")
    print("=" * 60)
    try:
        covariates = load_covariates_data()
        run_figure4_hte_public_service(sample, covariates)
        print(f"Saved: {FIGURE_DIR / 'figure_4.pdf'}")
    except Exception as e:
        print(f"Warning: Could not generate Figure 4: {e}")

    print("\n" + "=" * 60)
    print("Main text analysis complete!")
    print("=" * 60)

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## Balance Table


### Balance Table

Pre-treatment balance test comparing treated vs control municipalities (Table B.19.1)

In [ ]:
def run():
    """Run high-priority appendix B analyses."""
    ensure_output_dirs()

    print("Loading data...")
    electoral = load_electoral_data()
    sample = prepare_analysis_sample(electoral)

    # Table B.19.1: Balance table
    print("\n" + "=" * 60)
    print("Table B.19.1: Balance Table")
    print("=" * 60)
    try:
        covariates = load_covariates_data()
        balance_table = run_table_b19_balance(sample, covariates)
        print(balance_table)

        save_table_tex(
            balance_table,
            TABLE_DIR / "table_b19_1.tex",
            title="Balance Table: Pre-Treatment Covariates",
            note="Two-sample t-tests comparing treated vs never-treated municipalities.",
        )
    except Exception as e:
        print(f"Warning: Could not generate Table B.19.1: {e}")

    # Figure B.7.1: Suicide event study
    print("\n" + "=" * 60)
    print("Figure B.7.1: Suicide Event Study")
    print("=" * 60)
    try:
        demographics = load_demographics_data()
        demo_sample = prepare_analysis_sample(demographics)
        run_figure_b7_suicide_event_study(demo_sample)
        print(f"Saved: {FIGURE_DIR / 'figure_b7_1.pdf'}")
    except Exception as e:
        print(f"Warning: Could not generate Figure B.7.1: {e}")

    # Table B.4.1: Dose-response
    print("\n" + "=" * 60)
    print("Table B.4.1: Dose-Response Analysis")
    print("=" * 60)
    try:
        dose_table = run_table_b4_dose_response(sample)
        print(dose_table)

        save_table_tex(
            dose_table,
            TABLE_DIR / "table_b4_1.tex",
            title="Dose-Response Analysis: Months Since Infection",
            note="TWFE models using continuous intensity (months since infection) instead of binary treatment.",
        )
    except Exception as e:
        print(f"Warning: Could not generate Table B.4.1: {e}")

    print("\n" + "=" * 60)
    print("Appendix B analysis complete!")
    print("=" * 60)

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## Suicide Event Study


### Suicide Event Study

Effect of epidemic on suicide rates, testing mental health mechanism (Figure B.7.1)

In [ ]:
def run():
    """Run high-priority appendix B analyses."""
    ensure_output_dirs()

    print("Loading data...")
    electoral = load_electoral_data()
    sample = prepare_analysis_sample(electoral)

    # Table B.19.1: Balance table
    print("\n" + "=" * 60)
    print("Table B.19.1: Balance Table")
    print("=" * 60)
    try:
        covariates = load_covariates_data()
        balance_table = run_table_b19_balance(sample, covariates)
        print(balance_table)

        save_table_tex(
            balance_table,
            TABLE_DIR / "table_b19_1.tex",
            title="Balance Table: Pre-Treatment Covariates",
            note="Two-sample t-tests comparing treated vs never-treated municipalities.",
        )
    except Exception as e:
        print(f"Warning: Could not generate Table B.19.1: {e}")

    # Figure B.7.1: Suicide event study
    print("\n" + "=" * 60)
    print("Figure B.7.1: Suicide Event Study")
    print("=" * 60)
    try:
        demographics = load_demographics_data()
        demo_sample = prepare_analysis_sample(demographics)
        run_figure_b7_suicide_event_study(demo_sample)
        print(f"Saved: {FIGURE_DIR / 'figure_b7_1.pdf'}")
    except Exception as e:
        print(f"Warning: Could not generate Figure B.7.1: {e}")

    # Table B.4.1: Dose-response
    print("\n" + "=" * 60)
    print("Table B.4.1: Dose-Response Analysis")
    print("=" * 60)
    try:
        dose_table = run_table_b4_dose_response(sample)
        print(dose_table)

        save_table_tex(
            dose_table,
            TABLE_DIR / "table_b4_1.tex",
            title="Dose-Response Analysis: Months Since Infection",
            note="TWFE models using continuous intensity (months since infection) instead of binary treatment.",
        )
    except Exception as e:
        print(f"Warning: Could not generate Table B.4.1: {e}")

    print("\n" + "=" * 60)
    print("Appendix B analysis complete!")
    print("=" * 60)

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## Dose-Response Analysis


### Dose-Response Analysis

Continuous treatment intensity (months since infection) instead of binary (Table B.4.1)

In [ ]:
def run():
    """Run high-priority appendix B analyses."""
    ensure_output_dirs()

    print("Loading data...")
    electoral = load_electoral_data()
    sample = prepare_analysis_sample(electoral)

    # Table B.19.1: Balance table
    print("\n" + "=" * 60)
    print("Table B.19.1: Balance Table")
    print("=" * 60)
    try:
        covariates = load_covariates_data()
        balance_table = run_table_b19_balance(sample, covariates)
        print(balance_table)

        save_table_tex(
            balance_table,
            TABLE_DIR / "table_b19_1.tex",
            title="Balance Table: Pre-Treatment Covariates",
            note="Two-sample t-tests comparing treated vs never-treated municipalities.",
        )
    except Exception as e:
        print(f"Warning: Could not generate Table B.19.1: {e}")

    # Figure B.7.1: Suicide event study
    print("\n" + "=" * 60)
    print("Figure B.7.1: Suicide Event Study")
    print("=" * 60)
    try:
        demographics = load_demographics_data()
        demo_sample = prepare_analysis_sample(demographics)
        run_figure_b7_suicide_event_study(demo_sample)
        print(f"Saved: {FIGURE_DIR / 'figure_b7_1.pdf'}")
    except Exception as e:
        print(f"Warning: Could not generate Figure B.7.1: {e}")

    # Table B.4.1: Dose-response
    print("\n" + "=" * 60)
    print("Table B.4.1: Dose-Response Analysis")
    print("=" * 60)
    try:
        dose_table = run_table_b4_dose_response(sample)
        print(dose_table)

        save_table_tex(
            dose_table,
            TABLE_DIR / "table_b4_1.tex",
            title="Dose-Response Analysis: Months Since Infection",
            note="TWFE models using continuous intensity (months since infection) instead of binary treatment.",
        )
    except Exception as e:
        print(f"Warning: Could not generate Table B.4.1: {e}")

    print("\n" + "=" * 60)
    print("Appendix B analysis complete!")
    print("=" * 60)

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))